# CAP Topic Classification

Here I will investigate whether parliamentary cases can be classified into
Comparative Agendas Project (CAP) policy topics using semantic embeddings.

Previous exploratory analysis showed that E5 embeddings contain substantial
policy-related structure. K-means clustering recovered several recognizable
political domains, but the resulting clusters did not correspond one-to-one
with CAP major topics. Some clusters instead represented narrower subtopics or
cross-cutting semantic domains.

The aim of this notebook is therefore to move from unsupervised clustering to
a CAP-guided approach. CAP topic and subtopic descriptions are represented
semantically using multilingual E5 embeddings and compared with parliamentary
case embeddings. Existing human-coded CAP labels linked to ODA cases are used
for evaluation.

In [12]:
from pathlib import Path
import sys
import re
import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.project_data_ANEY import load_project_data

data = load_project_data()

df_questions = data["questions"]
df_cases = data["cases"]
df_cap_bills = data["cap_bills"]
df_cap_labels = data["cap_labels"]

print("Candidate questions:", df_questions.shape)
print("Parliamentary cases:", df_cases.shape)
print("CAP bills:", df_cap_bills.shape)
print("CAP-labelled ODA cases:", df_cap_labels.shape)

Candidate questions: (136, 2)
Parliamentary cases: (2128, 6)
CAP bills: (15101, 37)
CAP-labelled ODA cases: (533, 6)


## Parse the CAP codebook

In [13]:
CAP_DIR = PROJECT_ROOT / "data" / "temp_data" / "cap"

codebook_file = CAP_DIR / "topic_codebook.md"

rows = []

current_major_topic = None
current_major_name = None
current_subtopic = None
current_subtopic_name = None

for line in codebook_text.splitlines():
    stripped = line.strip()

    # Major topic, e.g. "1. Macroeconomics"
    major_match = re.match(r"^(\d+)\.\s+(.+)$", stripped)

    if major_match:
        current_major_topic = int(major_match.group(1))
        current_major_name = major_match.group(2)
        continue

    # Subtopic, e.g. "100: General"
    subtopic_match = re.match(r"^(\d+):\s+(.+)$", stripped)

    if subtopic_match:
        current_subtopic = int(subtopic_match.group(1))
        current_subtopic_name = subtopic_match.group(2)
        continue

    # Description belonging to the current subtopic
    if stripped.startswith("Description:") and current_subtopic is not None:
        description = stripped.removeprefix("Description:").strip()

        rows.append({
            "major_topic": current_major_topic,
            "major_name": current_major_name,
            "subtopic": current_subtopic,
            "subtopic_name": current_subtopic_name,
            "description": description
        })

df_cap_codebook = pd.DataFrame(rows)

df_cap_codebook.head(10)

,major_topic,major_name,subtopic,subtopic_name,description
0,1,Macroeconomics,100,General,Includes issues related to general domestic ma...
1,1,Macroeconomics,101,Interest Rates,"Includes issues related to inflation, cost of ..."
2,1,Macroeconomics,103,Unemployment Rate,Includes issues related to the unemployment ra...
3,1,Macroeconomics,104,Monetary Policy,Includes issues related to the monetary policy...
4,1,Macroeconomics,105,National Budget,"Issues related to public debt, budgeting, and ..."
5,1,Macroeconomics,107,Tax Code,"Includes issues related to tax policy, the imp..."
6,1,Macroeconomics,108,Industrial Policy,Includes issues related to manufacturing polic...
7,1,Macroeconomics,110,Price Control,Includes issues related to wage or price contr...
8,1,Macroeconomics,199,Other,Includes issues related to other macroeconomic...
9,2,Civil Rights,200,General,Includes issues related generally to civil rig...


In [14]:
print("Rows:", len(df_cap_codebook))
print("Major topics:", df_cap_codebook["major_topic"].nunique())
print("Subtopics:", df_cap_codebook["subtopic"].nunique())

print("\nSubtopics per major topic:")
print(
    df_cap_codebook
    .groupby(["major_topic", "major_name"])
    .size()
)

Rows: 212
Major topics: 21
Subtopics: 212

Subtopics per major topic:
major_topic  major_name           
1            Macroeconomics            9
2            Civil Rights             10
3            Health                   17
4            Agriculture               9
5            Labor                     9
6            Education                 9
7            Environment              11
8            Energy                    9
9            Immigration               1
10           Transportation            9
11           Law and Crime            13
12           Social Welfare            7
13           Housing                  11
14           Domestic Commerce        15
15           Defense                  17
16           Technology               10
17           Foreign Trade             8
18           International Affairs    12
19           Government Operations    18
20           Public Lands              7
21           Culture                   1
dtype: int64


## Create CAP topic representations

In [17]:
df_cap_codebook["text"] = (
    df_cap_codebook["major_name"]
    + ". "
    + df_cap_codebook["subtopic_name"]
    + ". "
    + df_cap_codebook["description"]
)

df_cap_codebook[
    ["major_topic", "subtopic", "text"]
].head()

,major_topic,subtopic,text
0,1,100,Macroeconomics. General. Includes issues relat...
1,1,101,Macroeconomics. Interest Rates. Includes issue...
2,1,103,Macroeconomics. Unemployment Rate. Includes is...
3,1,104,Macroeconomics. Monetary Policy. Includes issu...
4,1,105,Macroeconomics. National Budget. Issues relate...


In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

c:\Users\asket\Desktop\Environments\NLP\NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6030.44it/s]


In [19]:
cap_texts = [
    "passage: " + text
    for text in df_cap_codebook["text"]
]

cap_embeddings = model.encode(
    cap_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(cap_embeddings.shape)

Batches: 100%|██████████| 7/7 [00:00<00:00,  7.61it/s]

(212, 768)


In [20]:
df_cases["text"] = (
    df_cases["sag_titel"].fillna("")
    + " "
    + df_cases["sag_resume"].fillna("")
).str.strip()

case_texts = [
    "query: " + text
    for text in df_cases["text"]
]

case_embeddings = model.encode(
    case_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(case_embeddings.shape)

Batches: 100%|██████████| 67/67 [00:29<00:00,  2.23it/s]

(2128, 768)
